In [1]:
import sys
from pathlib import Path
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

# Caminho real do notebook 
NOTEBOOK_DIR = Path.cwd() 

# Sobe até a raiz do projeto (shopnow/) 
ROOT = NOTEBOOK_DIR.parents[2] 

# Adiciona ao sys.path 
if str(ROOT) not in sys.path: sys.path.append(str(ROOT)) 

# Agora sim, pode importar 
from pipeline.utils import find_repo_root, get_raw_dir

# 1. Configuração do ambiente

RAWDIR = get_raw_dir()

In [2]:
# 2. Configuração do Faker, Seed e geração dos dados

# Inicializa o Faker e a seed para reprodutibilidade
fake = Faker("pt_BR")
random.seed(42)

def gerar_atendimentos(qtd: int = 3000) -> list[dict]:
    """Gera uma lista de dicionários simulando atendimentos em nível enterprise."""

    canais = ["Site", "App", "Telefone", "WhatsApp", "Chat Online"]
    motivos = ["Entrega", "Pagamento", "Produto com defeito", "Troca", "Cancelamento"]
    submotivos = ["Atraso", "Erro na cobrança", "Produto errado", "Arrependimento", "Estorno"]
    prioridades = ["Baixa", "Média", "Alta", "Crítica"]
    areas = ["Logística", "Financeiro", "Comercial", "Tecnologia"]
    complexidades = ["Baixa", "Média", "Alta"]

    atendimentos = []

    for i in range(qtd):
        data_abertura = fake.date_time_between(start_date="-2y", end_date="now")
        
        # 80% dos casos são fechados
        fechado = random.choice([True, True, True, True, False])
        
        data_primeira_resposta = data_abertura + timedelta(
            hours=random.randint(1, 12)
        )

        if fechado:
            data_fechamento = data_abertura + timedelta(
                hours=random.randint(2, 96)
            )
        else:
            data_fechamento = None

        quantidade_reaberturas = random.choice([0, 0, 0, 1, 2])

        tempo_agente_minutos = random.randint(5, 90)
        custo_hora_agente = round(random.uniform(25, 80), 2)
        custo_atendimento = round((tempo_agente_minutos / 60) * custo_hora_agente, 2)

        valor_compensacao = random.choice([0, 0, 0, 20, 50, 100])
        valor_estornado = random.choice([0, 0, 0, 150, 300])

        nota_satisfacao = random.randint(1, 5)

        if nota_satisfacao >= 4:
            nps = "Promotor"
        elif nota_satisfacao == 3:
            nps = "Neutro"
        else:
            nps = "Detrator"

        atendimentos.append({
            # 🔑 Identificadores
            "id_atendimento": i + 1,
            "id_pedido": f"PED-{random.randint(10000,99999)}",
            "id_cliente": random.randint(1, 1500),
            "id_agente": random.randint(1, 50),

            # 📌 Classificação
            "canal": random.choice(canais),
            "motivo_principal": random.choice(motivos),
            "submotivo": random.choice(submotivos),
            "area_responsavel": random.choice(areas),
            "prioridade": random.choice(prioridades),
            "complexidade": random.choice(complexidades),

            # 📅 Datas
            "data_abertura": data_abertura,
            "data_primeira_resposta": data_primeira_resposta,
            "data_fechamento": data_fechamento,
            "quantidade_reaberturas": quantidade_reaberturas,

            # ⏱ Métricas de tempo (serão recalculadas no ETL)
            "tempo_agente_minutos": tempo_agente_minutos,

            # 💰 Custos
            "custo_hora_agente": custo_hora_agente,
            "custo_atendimento": custo_atendimento,
            "valor_compensacao": valor_compensacao,
            "valor_estornado": valor_estornado,

            # 😊 Qualidade
            "nota_satisfacao": nota_satisfacao,
            "nps_classificacao": nps,

            # 🎯 Estratégia
            "houve_reclamacao_publica": random.choice(["Sim", "Não"]),
            "cliente_churnou_apos_atendimento": random.choice(["Sim", "Não"]),
            "houve_recompra_30_dias": random.choice(["Sim", "Não"])
        })

    return atendimentos


In [3]:
# 3. Gera o DataFrame 
df_atendimento = pd.DataFrame(gerar_atendimentos(10000))
#df_atendimento

In [4]:
# 4. Salva o DataFrame como CSV no diretório de dados brutos
df_atendimento.to_csv(
    RAWDIR / "atendimento_raw.csv",
    index=False,
    encoding="utf-8"
)